# 04 — Dataset Acquisition

Builds a labeled training dataset for the card pre-grading model.

---

## ⚠ Domain Mismatch — Read This First

The pre-grading pipeline (notebooks 01–03) grades **raw, unslabbed card photos**
taken with a phone or camera. The right training data must match this domain.

| Source | Image type | Suitable for pre-grading? |
|--------|-----------|--------------------------|
| **PSA Cert API** | Flat professional scan of the card itself (no plastic case) | ✅ **Best** |
| Baseball dataset (Section 26) | Photos of the **PSA slab** (card inside plastic case) | ⚠ Grade classifier only |
| eBay Browse API (Section 28) | Photos of the **PSA slab** | ⚠ Grade classifier only |

**Bottom line:**
- **If you want to pre-grade raw cards** → only PSA cert scans (Section 27) are correct.
  They are flat scans of the actual card surface with no plastic obstruction.
- **Slab photos** are still useful for training a 5-bucket *grade classifier* (PSA 1–10)
  because visible wear is grade-correlated even through plastic — but expect a
  domain gap when you run it on raw card photos.

---

| Step | Source | Images | Auth needed | Suitable for |
|------|--------|--------|-------------|--------------|
| **26** | PSA-Grades-Baseball (slab photos) | 11,500 | None — downloads now | Grade classifier (not pre-grading) |
| **27** | PSA Cert API | Up to 50/day free | Free PSA account | Pre-grading ✅ |
| **28** | eBay Browse API (slab photos) | Hundreds/run | `EBAY_APP_ID` | Grade classifier (not pre-grading) |
| **29** | Merge + stats + train | All above | — | — |

## Setup

In [1]:
# ── Install dependencies ─────────────────────────────────────────
import subprocess, sys
packages = ['opencv-python','numpy','matplotlib','requests','tqdm']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

import cv2, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

# GRADE_COLORS for the distribution plot (mirrors grading_utils)
GRADE_COLORS = {
    10:'#22c55e', 9:'#4ade80', 8:'#86efac',
     7:'#fbbf24', 6:'#fb923c', 5:'#f97316',
     4:'#ef4444', 3:'#dc2626', 2:'#b91c1c', 1:'#7f1d1d',
}

# PSA bucket mapping (mirrors 03 / resnet pipeline)
PSA_TO_BUCKET  = {1:0,2:0, 3:1,4:1, 5:2,6:2, 7:3,8:3, 9:4,10:4}
BUCKET_NAMES   = ['Poor','Very Good','Excellent','Near Mint','Mint']

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi']      = 120
plt.rcParams['axes.facecolor']  = '#0d1117'
plt.rcParams['figure.facecolor']= '#0d1117'
plt.rcParams['text.color']      = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color']     = 'white'
plt.rcParams['ytick.color']     = 'white'

print('✅ Dataset acquisition notebook ready')

✅ Dataset acquisition notebook ready


## 26. Step 1 — Download Open Datasets (Slab Photos)

Downloads the `samsilverman/PSA-Grades-Baseball` dataset: 11,500 baseball card
images perfectly balanced across PSA grades 1–10.

**What these images are:** eBay listing photos of graded baseball cards **inside
PSA plastic cases**. Grade label = folder name (`psa1/` … `psa10/`).

**Use for:** Training a grade *classifier* model. Not for the pre-grading
pipeline which expects raw card photos.

In [ ]:
import subprocess
from pathlib import Path

DATASET_ROOT = Path('datasets')
DATASET_ROOT.mkdir(exist_ok=True)

# ── PSA-Grades-Baseball (11,500 slab photos, balanced grades 1-10) ─
BASEBALL_DIR = DATASET_ROOT / 'PSA-Grades-Baseball'
if not BASEBALL_DIR.exists():
    print('Cloning PSA-Grades-Baseball (~150 MB) …')
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/samsilverman/PSA-Grades-Baseball.git',
        str(BASEBALL_DIR),
    ], check=True)
    print('✅ Done')
else:
    print(f'✅ Already present: {BASEBALL_DIR}')

total = 0
for grade in range(1, 11):
    folder = BASEBALL_DIR / f'psa{grade}'
    n = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
    total += n
    bar = '█' * (n // 115)
    print(f'  PSA {grade:>2}: {n:>5} images  {bar}')
print(f'  Total : {total:>5} images')
print()
print('⚠ These are slab photos (card inside PSA plastic case).')
print('  Use for grade classification — not for raw-card pre-grading.')

## 27. Step 2 — PSA Cert API Scraper ✅ Best for Pre-Grading

PSA cert scans are **flat, professional images of the card surface itself**
with no slab or plastic — the same domain as a user photographing a raw card.

**Endpoint:** `https://api.psacard.com/publicapi/cert/GetByCertNumber/{cert}`  
**Auth:** Free account at [psacard.com/publicapi](https://www.psacard.com/publicapi)  
**Rate limit:** 100 API calls / day free ≈ **50 certs/day** (detail + image = 2 calls per cert)  
**Image availability:** Only certs graded **after October 2021** have photos

### Strategy
Cert numbers are sequential 8-digit integers. The scraper:
1. Takes an **anchor cert** from any slab you own (scan the barcode label)
2. Scans ± `scan_radius` certs from that anchor, filtering for `"Pokemon"` brand
3. Tracks daily quota in `quota.json` — safe to run every day, resumes automatically

### Yield estimate
Pokémon is ~5–10 % of PSA's volume. With `scan_radius=5000`, roughly 500–1000
of the 10,000 scanned certs will be Pokémon, giving **250–500 images/day**
once you account for the 50-cert API limit.

To get more: run once per day, or upgrade to a paid PSA API plan.

In [ ]:
import os, json as _json, time, requests
from pathlib import Path
from datetime import date

# ── CONFIG — fill these in ────────────────────────────────────────
PSA_TOKEN    = os.getenv('PSA_API_TOKEN', '')   # get from psacard.com/publicapi
ANCHOR_CERT  = ''          # a cert number you own, e.g. '12345678'
SCAN_RADIUS  = 5_000       # scan ± this many certs from anchor
TARGET_BRAND = 'Pokemon'   # filter: only save cards matching this brand
SAVE_DIR     = Path('datasets/pokemon_psa')
DAILY_LIMIT  = 90          # stay under the 100-call/day free tier
# ─────────────────────────────────────────────────────────────────

SAVE_DIR.mkdir(parents=True, exist_ok=True)
QUOTA_FILE = SAVE_DIR / 'quota.json'
META_FILE  = SAVE_DIR / 'metadata.csv'


def _load_quota() -> dict:
    """Load today's call counter, reset if it's a new day."""
    today = str(date.today())
    if QUOTA_FILE.exists():
        q = _json.loads(QUOTA_FILE.read_text())
        if q.get('date') == today:
            return q
    return {'date': today, 'calls': 0}


def _save_quota(q: dict):
    QUOTA_FILE.write_text(_json.dumps(q))


def _psa_get(path: str, token: str) -> dict | None:
    """Single authenticated GET to the PSA API; returns parsed JSON or None."""
    url = f'https://api.psacard.com/publicapi/{path}'
    try:
        r = requests.get(url, headers={'authorization': f'bearer {token}'}, timeout=10)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            print('  ⚠ Rate-limited — stopping for today')
            return 'RATE_LIMIT'
    except requests.RequestException as e:
        print(f'  Network error: {e}')
    return None


def _download_image(url: str, dest: Path) -> bool:
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            dest.write_bytes(r.content)
            return True
    except requests.RequestException:
        pass
    return False


def fetch_cert(cert_num: int, token: str) -> dict | None:
    """
    Look up one cert: returns metadata dict with image paths,
    None if not found / not the target brand, 'RATE_LIMIT' if quota hit.
    """
    cert_str = str(cert_num)

    # ── Details call ─────────────────────────────────────────────
    data = _psa_get(f'cert/GetByCertNumber/{cert_str}', token)
    if data == 'RATE_LIMIT':
        return 'RATE_LIMIT'
    if not data:
        return None

    cert = data.get('PSACert', {})
    brand = cert.get('Brand', '') or cert.get('Variety', '')
    grade_raw = cert.get('Grade', '') or cert.get('CardGrade', '')

    # Filter by brand
    if TARGET_BRAND.lower() not in brand.lower():
        return None

    try:
        grade = int(str(grade_raw).split('.')[0])
        if not (1 <= grade <= 10):
            return None
    except (ValueError, TypeError):
        return None

    # ── Images call ──────────────────────────────────────────────
    imgs = _psa_get(f'cert/GetImagesByCertNumber/{cert_str}', token)
    if imgs == 'RATE_LIMIT':
        return 'RATE_LIMIT'
    if not imgs:
        return None

    # Save front image (and back if available)
    saved_paths = {}
    for img_obj in (imgs if isinstance(imgs, list) else []):
        url    = img_obj.get('ImageURL', '')
        is_front = img_obj.get('IsFrontImage', True)
        if not url:
            continue
        side  = 'front' if is_front else 'back'
        dest  = SAVE_DIR / f'psa{grade}' / f'{cert_str}_{side}.jpg'
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            _download_image(url, dest)
        saved_paths[side] = str(dest)

    if not saved_paths:
        return None

    return {
        'cert':    cert_str,
        'grade':   grade,
        'brand':   brand,
        'subject': cert.get('Subject', ''),
        'year':    cert.get('Year', ''),
        'set':     cert.get('Variety', ''),
        'card_no': cert.get('CardNumber', ''),
        **{f'path_{k}': v for k, v in saved_paths.items()},
    }


def scrape_psa(anchor_cert: str = ANCHOR_CERT,
               scan_radius: int = SCAN_RADIUS,
               token: str       = PSA_TOKEN,
               max_new: int      = 200) -> list[dict]:
    """
    Scrape PSA certs around an anchor cert number.

    anchor_cert : a cert you already know (any slab you own)
    scan_radius : how many certs above/below to check
    max_new     : stop after saving this many new images in this run
    """
    if not token:
        print('⚠ PSA_API_TOKEN not set.')
        print('  1. Create a free account at https://www.psacard.com/publicapi')
        print('  2. Copy your bearer token')
        print('  3. Set PSA_TOKEN = "your_token_here" in the CONFIG block above')
        return []

    if not anchor_cert:
        print('⚠ ANCHOR_CERT not set.')
        print('  Look at any PSA slab you own — the cert number is on the label.')
        print('  Set ANCHOR_CERT = "12345678" in the CONFIG block above.')
        return []

    # Load existing metadata so we can skip already-done certs
    done_certs = set()
    if META_FILE.exists():
        import csv
        with open(META_FILE, newline='') as f:
            done_certs = {row['cert'] for row in csv.DictReader(f)}
    print(f'Already collected: {len(done_certs)} certs')

    quota    = _load_quota()
    results  = []
    new_count = 0
    anchor    = int(anchor_cert)

    # Interleave above/below anchor: anchor, anchor+1, anchor-1, anchor+2, ...
    offsets = [o for d in range(scan_radius+1) for o in ([d, -d] if d else [0])]

    for offset in offsets:
        cert_num = anchor + offset
        if cert_num < 1:
            continue
        cert_str = str(cert_num)

        if cert_str in done_certs:
            continue
        if new_count >= max_new:
            print(f'Reached max_new={max_new} — stopping this run')
            break
        if quota['calls'] >= DAILY_LIMIT:
            print(f'Daily quota ({DAILY_LIMIT}) reached — resume tomorrow')
            break

        quota['calls'] += 2   # detail + image = 2 calls
        _save_quota(quota)

        result = fetch_cert(cert_num, token)

        if result == 'RATE_LIMIT':
            quota['calls'] = DAILY_LIMIT
            _save_quota(quota)
            break
        if result:
            results.append(result)
            new_count += 1
            g = result['grade']
            print(f'  ✅ PSA {g:>2}  {result["subject"]:30}  '
                  f'cert {cert_str}  (quota {quota["calls"]}/{DAILY_LIMIT})')

        time.sleep(0.4)   # be respectful

    # Append to metadata CSV
    if results:
        import csv
        fieldnames = ['cert','grade','brand','subject','year','set',
                      'card_no','path_front','path_back']
        write_header = not META_FILE.exists()
        with open(META_FILE, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            if write_header:
                w.writeheader()
            w.writerows(results)

    # Summary
    grade_counts = {}
    for r in results:
        grade_counts[r['grade']] = grade_counts.get(r['grade'], 0) + 1
    print(f'\n✅ Saved {new_count} new certs this run')
    print(f'   Quota used today: {quota["calls"]} / {DAILY_LIMIT}')
    if grade_counts:
        print('   By grade:', dict(sorted(grade_counts.items())))

    return results


# ── Single cert test ─────────────────────────────────────────────
def test_psa_connection(cert: str, token: str = PSA_TOKEN):
    """Quick sanity check — look up one cert and print what comes back."""
    if not token:
        print('Set PSA_TOKEN first'); return
    print(f'Testing cert {cert} …')
    data = _psa_get(f'cert/GetByCertNumber/{cert}', token)
    if data and data != 'RATE_LIMIT':
        c = data.get('PSACert', {})
        print(f'  Brand  : {c.get("Brand")}')
        print(f'  Subject: {c.get("Subject")}')
        print(f'  Grade  : {c.get("Grade")}')
        print(f'  Year   : {c.get("Year")}')
    else:
        print('  No data returned — check your token')

    imgs = _psa_get(f'cert/GetImagesByCertNumber/{cert}', token)
    if isinstance(imgs, list):
        print(f'  Images : {len(imgs)} ({["front" if i.get("IsFrontImage") else "back" for i in imgs]})')
    else:
        print(f'  Images : none (card may pre-date Oct 2021 or token issue)')


print('✅ PSA scraper ready')
print()
print('Quick start:')
print('  1. Set PSA_TOKEN  = "your_bearer_token"  (psacard.com/publicapi)')
print('  2. Set ANCHOR_CERT = "12345678"           (cert # from any slab you own)')
print('  3. test_psa_connection(ANCHOR_CERT)')
print('  4. scrape_psa(max_new=40)                (uses ~80 of your 100 daily calls)')

## 28. Step 3 — eBay Browse API Collector (Slab Photos)

Downloads images from **active eBay listings** of graded Pokémon cards.
Grade label is parsed from the listing title via regex
(`"PSA 9 1999 Pokemon Charizard"` → grade 9).

**What these images are:** Seller photos of **slabbed cards** (card inside
PSA/BGS plastic case). Same domain caveat as Section 26 — useful for
grade classification, not for raw-card pre-grading.

**Requires:** `EBAY_APP_ID` + `EBAY_CERT_ID`
(from the same app registered at [developer.ebay.com](https://developer.ebay.com)).

### Noise sources to be aware of
| Issue | Frequency | Impact |
|-------|-----------|--------|
| Mislabelled grade in title | ~5 % | Low — most sellers label correctly |
| Wrong PSA company (BGS/SGC listed as PSA) | ~3 % | Low |
| Slab shot from an angle (obscures card) | ~15 % | Medium |
| Multiple cards in one listing | ~2 % | Low | 

In [ ]:
import os, re, time, requests
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────
EBAY_APP_ID    = os.getenv('EBAY_APP_ID', '')
EBAY_SAVE_DIR  = Path('datasets/pokemon_ebay')
EBAY_MAX_PAGES = 5        # 200 results per search term (5 pages × 40 items)
EBAY_SLEEP     = 0.5      # seconds between image downloads

# Search terms — each produces a separate batch
# Format: (search_query, expected_grade)
# Grade is extracted from title but query guides what PSA grade to fetch
EBAY_SEARCHES = [
    ('PSA 10 pokemon charizard',   10),
    ('PSA 10 pokemon pikachu',     10),
    ('PSA 9 pokemon charizard',     9),
    ('PSA 9 pokemon blastoise',     9),
    ('PSA 9 pokemon venusaur',      9),
    ('PSA 8 pokemon base set',      8),
    ('PSA 7 pokemon base set',      7),
    ('PSA 6 pokemon base set',      6),
    ('PSA 5 pokemon base set',      5),
    ('PSA 4 pokemon base set',      4),
    ('PSA 3 pokemon base set',      3),
    ('PSA 2 pokemon',               2),
    ('PSA 1 pokemon poor',          1),
]

# ─────────────────────────────────────────────────────────────────
EBAY_SAVE_DIR.mkdir(parents=True, exist_ok=True)


def _ebay_oauth_token(app_id: str) -> str | None:
    """
    Get a client-credentials OAuth token from eBay.
    Requires EBAY_CERT_ID env var (from your eBay developer app).
    """
    cert_id = os.getenv('EBAY_CERT_ID', '')
    if not app_id or not cert_id:
        return None
    resp = requests.post(
        'https://api.ebay.com/identity/v1/oauth2/token',
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        auth=(app_id, cert_id),
        data='grant_type=client_credentials'
             '&scope=https://api.ebay.com/oauth/api_scope',
        timeout=10,
    )
    if resp.status_code == 200:
        return resp.json().get('access_token')
    return None


GRADE_RE = re.compile(
    r'\bpsa\s*(?:grade[d]?\s*)?([1-9]|10)\b',
    re.IGNORECASE,
)

def _parse_grade(title: str) -> int | None:
    """Extract PSA grade integer from a listing title."""
    m = GRADE_RE.search(title)
    if m:
        g = int(m.group(1))
        if 1 <= g <= 10:
            return g
    return None


def _browse_search(query: str, token: str,
                   limit: int = 40, offset: int = 0) -> list[dict]:
    """Single eBay Browse API search call."""
    resp = requests.get(
        'https://api.ebay.com/buy/browse/v1/item_summary/search',
        headers={
            'Authorization': f'Bearer {token}',
            'X-EBAY-C-MARKETPLACE-ID': 'EBAY_US',
            'Content-Type': 'application/json',
        },
        params={
            'q':       query,
            'limit':   limit,
            'offset':  offset,
            'filter':  'conditionIds:{1000}',   # 1000 = New/Graded slab
        },
        timeout=15,
    )
    if resp.status_code == 200:
        return resp.json().get('itemSummaries', [])
    return []


def collect_ebay_images(searches: list = EBAY_SEARCHES,
                        max_pages: int  = EBAY_MAX_PAGES,
                        app_id: str     = EBAY_APP_ID) -> list[dict]:
    """
    Run each search query, parse grade from title, download the main
    listing image, and save to datasets/pokemon_ebay/psa{grade}/.
    """
    if not app_id:
        print('⚠ EBAY_APP_ID not set.')
        print('  Set EBAY_APP_ID = "your_app_id" in the CONFIG block above,')
        print('  or export it as an environment variable.')
        return []

    token = _ebay_oauth_token(app_id)
    if not token:
        print('⚠ Could not get eBay OAuth token.')
        print('  Also set EBAY_CERT_ID env var (from developer.ebay.com app page).')
        return []

    # Load already-downloaded item IDs to skip
    done_ids_file = EBAY_SAVE_DIR / 'downloaded_ids.txt'
    done_ids = set(done_ids_file.read_text().splitlines())                if done_ids_file.exists() else set()

    results  = []
    new_ids  = []

    for query, fallback_grade in searches:
        print(f'\nSearching: {query!r}')
        for page in range(max_pages):
            items = _browse_search(query, token,
                                   limit=40, offset=page * 40)
            if not items:
                break

            for item in items:
                item_id = item.get('itemId', '')
                if item_id in done_ids:
                    continue

                title  = item.get('title', '')
                grade  = _parse_grade(title) or fallback_grade
                img_url = (item.get('image') or {}).get('imageUrl', '')

                if not img_url:
                    continue

                dest_dir = EBAY_SAVE_DIR / f'psa{grade}'
                dest_dir.mkdir(parents=True, exist_ok=True)
                dest = dest_dir / f'{item_id}.jpg'

                if not dest.exists():
                    try:
                        r = requests.get(img_url, timeout=10)
                        if r.status_code == 200:
                            dest.write_bytes(r.content)
                    except requests.RequestException:
                        continue

                results.append({'item_id': item_id, 'grade': grade,
                                 'title': title, 'path': str(dest)})
                new_ids.append(item_id)
                time.sleep(EBAY_SLEEP)

            time.sleep(0.3)   # between pages

    # Persist downloaded IDs
    if new_ids:
        with open(done_ids_file, 'a') as f:
            f.write('\n'.join(new_ids) + '\n')

    # Summary
    from collections import Counter
    grade_dist = Counter(r['grade'] for r in results)
    print(f'\n✅ Collected {len(results)} new eBay images this run')
    print('   Grade distribution:', dict(sorted(grade_dist.items())))
    return results


print('✅ eBay collector ready')
print()
print('Usage:')
print('  Set EBAY_APP_ID  = "your_app_id"')
print('  Set EBAY_CERT_ID env var (same app on developer.ebay.com)')
print('  results = collect_ebay_images()')

## 29. Step 4 — Merge, Domain Check & Train

Combines all sources, prints a distribution report, checks for suspicious
images (slab vs flat scan), and launches ResNet-18 training.

In [ ]:
import shutil, csv
from pathlib import Path
from collections import Counter, defaultdict

UNIFIED_DIR = Path('datasets/unified')


def check_is_slab(img_path: str) -> bool:
    """
    Heuristic: PSA slab photos have a thick, very uniform border (the yellow
    label band) and rounded-corner aspect around the card.

    Looks for a high proportion of near-pure yellow pixels (PSA label color)
    in the bottom 20% of the image.

    Returns True if image appears to be a slab photo.
    """
    try:
        import cv2, numpy as np
        img = cv2.imread(str(img_path))
        if img is None: return False
        h, w = img.shape[:2]
        bottom = img[int(h*0.75):, :]          # bottom 25%
        hsv = cv2.cvtColor(bottom, cv2.COLOR_BGR2HSV)
        # PSA yellow label: hue 20-35, sat >100, val >150
        yellow_mask = (
            (hsv[:,:,0] >= 18) & (hsv[:,:,0] <= 38) &
            (hsv[:,:,1] > 80)  &
            (hsv[:,:,2] > 120)
        )
        yellow_frac = yellow_mask.sum() / yellow_mask.size
        return bool(yellow_frac > 0.08)
    except Exception:
        return False


def merge_datasets(sources: list = None,
                   dest: Path = UNIFIED_DIR,
                   min_dim: int = 100) -> dict:
    """Merge all source folders into dest/psa{1-10}/ with per-source stats."""
    if sources is None:
        sources = [
            Path('datasets/PSA-Grades-Baseball'),
            Path('datasets/pokemon_psa'),
            Path('datasets/pokemon_ebay'),
        ]

    dest.mkdir(parents=True, exist_ok=True)
    counts  = defaultdict(lambda: defaultdict(int))
    slab_counts = defaultdict(int)
    skipped = 0

    for src_root in sources:
        src_name = src_root.name
        if not src_root.exists():
            print(f'  ⚠ Skipping {src_root} (not found)')
            continue

        for grade in range(1, 11):
            for folder_name in (f'psa{grade}', f'grade_{grade}', str(grade)):
                folder = src_root / folder_name
                if not folder.exists(): continue
                for img_path in sorted(folder.glob('*')):
                    if img_path.suffix.lower() not in ('.jpg','.jpeg','.png','.webp'):
                        continue
                    try:
                        import cv2 as _cv2
                        im = _cv2.imread(str(img_path))
                        if im is None or min(im.shape[:2]) < min_dim:
                            skipped += 1; continue
                    except Exception:
                        pass
                    dest_folder = dest / f'psa{grade}'
                    dest_folder.mkdir(exist_ok=True)
                    target = dest_folder / f'{src_name}_{img_path.name}'
                    if not target.exists():
                        shutil.copy2(img_path, target)
                    counts[src_name][grade] += 1
                    if check_is_slab(str(img_path)):
                        slab_counts[src_name] += 1

    # ── Stats ────────────────────────────────────────────────────
    all_by_grade = defaultdict(int)
    for src, gc in counts.items():
        for g, n in gc.items():
            all_by_grade[g] += n

    total = sum(all_by_grade.values())
    print(f'\n{"━"*60}')
    print(f'  Unified dataset → {dest}')
    print(f'{"━"*60}')
    for grade in range(1, 11):
        n   = all_by_grade[grade]
        bar = '█' * (n // max(total // 200, 1))
        print(f'  PSA {grade:>2}  {n:>6} images  {bar}')
    print(f'{"─"*60}')
    print(f'  Total            {total:>6} images')
    if skipped: print(f'  Skipped          {skipped:>6}')
    print()

    print(f'  {"Source":<35} {"Images":>7}  {"Slab %":>7}  Domain')
    print(f'  {"-"*60}')
    for src, gc in counts.items():
        n    = sum(gc.values())
        slabs = slab_counts.get(src, 0)
        slab_pct = slabs / max(n, 1) * 100
        domain = '⚠ slab photos' if slab_pct > 30 else '✅ flat scans'
        print(f'  {src:<35} {n:>7}  {slab_pct:>6.0f}%  {domain}')

    if slab_counts:
        total_slabs = sum(slab_counts.values())
        print(f'\n  ⚠ {total_slabs}/{total} images ({total_slabs/max(total,1):.0%}) appear to be slab photos.')
        print(  '    These are fine for a grade classifier.')
        print(  '    For pre-grading raw cards, prioritise Section 27 (PSA cert scans).')

    return dict(all_by_grade)


def plot_dataset_distribution(by_grade: dict):
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')
    grades = list(range(1, 11))
    counts_list = [by_grade.get(g, 0) for g in grades]
    colors = [GRADE_COLORS.get(g, '#888') for g in grades]
    bars = ax.bar(grades, counts_list, color=colors, width=0.7, edgecolor='#30363d')
    ax.set_xticks(grades)
    ax.set_xticklabels([f'PSA {g}' for g in grades], color='white', fontsize=9)
    ax.set_ylabel('Images', color='white')
    ax.set_title(f'Training Dataset — {sum(counts_list):,} images total', color='white', fontsize=12)
    ax.tick_params(colors='white')
    for bar, n in zip(bars, counts_list):
        if n:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                    str(n), ha='center', color='white', fontsize=8)
    plt.tight_layout(); plt.show()


def train_val_split(unified_dir: Path = UNIFIED_DIR,
                    val_frac: float = 0.10) -> tuple:
    import random
    rng = random.Random(42)
    train_lines, val_lines = [], []
    for grade in range(1, 11):
        folder = unified_dir / f'psa{grade}'
        if not folder.exists(): continue
        paths = [str(p) for p in sorted(folder.glob('*')) if p.is_file()]
        rng.shuffle(paths)
        split = max(1, int(len(paths) * val_frac))
        val_lines   += [f'{p}\t{grade}' for p in paths[:split]]
        train_lines += [f'{p}\t{grade}' for p in paths[split:]]
    (unified_dir / 'train.txt').write_text('\n'.join(train_lines))
    (unified_dir / 'val.txt').write_text('\n'.join(val_lines))
    print(f'✅ train.txt: {len(train_lines)} images')
    print(f'   val.txt  : {len(val_lines)} images')
    return len(train_lines), len(val_lines)


# ── Run ───────────────────────────────────────────────────────────
by_grade = merge_datasets()

if sum(by_grade.values()) > 0:
    plot_dataset_distribution(by_grade)
    n_train, n_val = train_val_split()
    if n_train >= 100:
        print('\nLaunch training in 03_saliency_explainability.ipynb:')
        print('  model, history = train_resnet_grader(')
        print('      image_dir   = "datasets/unified",')
        print('      num_classes = 5,')
        print('      epochs      = 10,')
        print('  )')
    else:
        print(f'\nNeed more images (have {n_train}). Run Section 27 daily.')
else:
    print('No images yet — run Sections 26-28 first.')